# Silver layer

Preprocessing for text cleaning is as followed:  
- Normalize line endings
- Replace tabs and non-breaking spaces
- Remove zero-width spaces if present
- Remove trailing spaces on lines
- Remove repeated spaces inside lines
- Normalize spaces around line breaks
- Reduce too many blank lines
- Remove very common PDF page markers if present

In [53]:
# Import needed libraries
import pandas as pd
import re

# Import bronze dataframe + sanity checks

In [54]:
# Load bronze data
project_data = pd.read_json("../Data/bronze/doc_01_bronze.json", orient='records')
print(project_data.head())


  document_id                                           raw_text
0      doc_01  Smart Delta Water Reuse\n\nFreshwater resource...


# Clean incoming text data

First we will Normalize line endings

In [55]:
def normalize_line_endings(text):
    return text.replace("\r\n", "\n").replace("\r", "\n")
    
#   print(normalize_line_endings(project_data['raw_text']))

Replace tabs and non-breaking spaces

In [56]:
def replace_tab_non_breaking_space(text):
    text = text.replace("\t", " ")
    text = text.replace("\xa0", " ")
    return text

#   print(replace_tab_non_breaking_space(project_data['raw_text']))

Remove zero-width spaces if present

In [57]:
def remove_zero_width_characters(text):
    zero_width_chars = [
        '\u200B',  # Zero Width Space
        '\u200C',  # Zero Width Non-Joiner
        '\u200D',  # Zero Width Joiner
        '\uFEFF'   # Zero Width No-Break Space
    ]
    for char in zero_width_chars:
        text = text.replace(char, '')
    return text

#   print(remove_zero_width_characters(project_data['raw_text']))

Remove trailing spaces on lines

In [58]:
def remove_trailing_spaces(text):
    return re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)
#   print(remove_trailing_spaces(project_data['raw_text']))

Remove repeated spaces inside lines

In [59]:
def remove_repeated_spaces(text):
    return re.sub(r"[ ]{2,}", " ", text)

Normalize spaces around line breaks

In [60]:
def normalize_spaces_around_line_breaks(text):
    return re.sub(r" *\n *", "\n", text)

#   print(normalize_spaces(project_data['raw_text']))

Reduce too many blank lines

In [61]:
def remove_many_blank_lines(text):
    return re.sub(r"\n{3,}", "\n\n", text)


#   print(remove_many_blank_lines(project_data['raw_text']))

Remove very common PDF page markers if present

In [62]:
def remove_common_pdf_artifacts(text):
    text = re.sub(r"-\n", "", text)  # fix hyphenated line breaks
    text = re.sub(r"Page \d+ of \d+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^Page \d+$", "", text, flags=re.IGNORECASE | re.MULTILINE)
    return text

#   print(remove_common_pdf_artifacts(project_data['raw_text']))

Run all functions on text

In [63]:
def clean_text(text):
    text = normalize_line_endings(text)
    text = replace_tab_non_breaking_space(text)
    text = remove_zero_width_characters(text)
    text = remove_trailing_spaces(text)
    text = remove_repeated_spaces(text)
    text = normalize_spaces_around_line_breaks(text)
    text = remove_common_pdf_artifacts(text)
    text = remove_many_blank_lines(text)
    return text.strip()


Run sanity checks

In [ ]:
def sanity_check(text):
    print("Original Text:\n")
    print(text[:500])
    print("\n---\n")
    print("Cleaned Text:\n")
    cleaned = clean_text(text)
    print(cleaned[:500])
    return cleaned

print(sanity_check(project_data['raw_text'][0]))

Original Text:

'Smart Delta Water Reuse\n\nFreshwater resources in the southwestern Netherlands are under increasing pressure. Periods of drought occur more frequently, while agriculture, industry and urban development continue to demand large quantities of water. In Zeeland, this problem is intensified by salinization, which limits the availability of freshwater for irrigation and industrial processes.\n\nAt the same time, large volumes of treated wastewater are discharged into rivers and coastal waters. Although '

---


Cleaned Text:

Smart Delta Water Reuse

Freshwater resources in the southwestern Netherlands are under increasing pressure. Periods of drought occur more frequently, while agriculture, industry and urban development continue to demand large quantities of water. In Zeeland, this problem is intensified by salinization, which limits the availability of freshwater for irrigation and industrial processes.

At the same time, large volumes of treated wastewater are dischar

In [65]:
# save the cleaned data to a new json file
project_data['cleaned_text'] = project_data['raw_text'].apply(clean_text)
project_data.to_json("../Data/silver/doc_01_silver.json", orient='records', lines=True)